# Tópico 2 - Atividades de Otimização
**UFAM - PPGEE | Prof. Kenny Vinente dos Santos, Dr. | Março 2026**

Aluno: Alessandro da Silva Maciel (aluno especial)

## 1.1 — (a) Gradient Descent e (b) Newton's Method

$f(x_1,x_2,x_3) = x_3\ln(e^{x_1/x_3}+e^{x_2/x_3}) + (x_3-2)^2 + e^{1/(x_1+x_2)}$

dom $f = \{x \in \mathbb{R}^3 : x_1+x_2>0,\; x_3>0\}$

Backtracking line search: $t_{init}=1$, $\alpha=0.4$, $\beta=0.5$. Tolerância: $\|\nabla f\| < 10^{-5}$.

In [4]:
using LinearAlgebra, Printf

# ---------- Função, gradiente (dif. finitas), hessiana (dif. finitas) ----------
f(x) = x[3]*log(exp(x[1]/x[3]) + exp(x[2]/x[3])) + (x[3]-2)^2 + exp(1/(x[1]+x[2]))
in_dom(x) = (x[1]+x[2]) > 1e-10 && x[3] > 1e-10

function grad_f(x; h=1e-5)
    fx = f(x); g = zeros(3)
    for i in 1:3; xp = copy(x); xp[i] += h; g[i] = (f(xp)-fx)/h; end
    return g
end

function hess_f(x; h=1e-5)
    H = zeros(3,3)
    for i in 1:3
        xp = copy(x); xm = copy(x); xp[i] += h; xm[i] -= h
        H[:,i] = (grad_f(xp;h=h) - grad_f(xm;h=h)) / (2h)
    end
    return 0.5(H + H')
end

# ---------- Backtracking line search ----------
function backtrack(x, d, g; t0=1.0, α=0.4, β=0.5)
    t, fx, gd = t0, f(x), dot(g,d)
    for _ in 1:100
        xn = x + t*d
        in_dom(xn) && f(xn) <= fx + α*t*gd && return t
        t *= β
    end
    return t
end

# ---------- (a) Gradient Descent ----------
function gradient_descent(x0; tol=1e-5, maxiter=10000)
    x = copy(x0); hist = [(copy(x), f(x), norm(grad_f(x)))]
    for k in 1:maxiter
        g = grad_f(x); norm(g) < tol && break
        t = backtrack(x, -g, g); x = x - t*g
        push!(hist, (copy(x), f(x), norm(grad_f(x))))
    end
    return x, hist
end

# ---------- (b) Newton's Method ----------
function newton_method(x0; tol=1e-5, maxiter=1000)
    x = copy(x0); hist = [(copy(x), f(x), norm(grad_f(x)))]
    for k in 1:maxiter
        g = grad_f(x); norm(g) < tol && break
        H = hess_f(x)
        d = try -H\g catch; -g end
        dot(g,d) >= 0 && (d = -g)
        t = backtrack(x, d, g); x = x + t*d
        push!(hist, (copy(x), f(x), norm(grad_f(x))))
    end
    return x, hist
end

# ---------- Execução ----------
x0 = [1.0, 1.0, 2.0]
x_gd, h_gd = gradient_descent(x0)
x_nt, h_nt = newton_method(x0)

function print_hist(label, hist; show_all=false)
    println("="^65)
    println("  $label")
    println("="^65)
    @printf("  Ponto inicial : [%.4f, %.4f, %.4f]\n", x0...)
    println("-"^65)
    @printf("  %5s  %14s  %14s\n", "Iter", "f(x)", "||∇f(x)||")
    println("-"^65)
    if show_all || length(hist) <= 12
        for (i,h) in enumerate(hist)
            @printf("  %5d  %14.6f  %14.2e\n", i-1, h[2], h[3])
        end
    else
        for i in 1:5
            @printf("  %5d  %14.6f  %14.2e\n", i-1, hist[i][2], hist[i][3])
        end
        println("     ...")
        for i in max(6,length(hist)-2):length(hist)
            @printf("  %5d  %14.6f  %14.2e\n", i-1, hist[i][2], hist[i][3])
        end
    end
    println("-"^65)
end

print_hist("1.1(a) GRADIENT DESCENT — Backtracking (t₀=1, α=0.4, β=0.5)", h_gd)
@printf("  x*        = [%.6f, %.6f, %.6f]\n", x_gd...)
@printf("  f(x*)     = %.6f\n", f(x_gd))
@printf("  ||∇f(x*)|| = %.2e\n", norm(grad_f(x_gd)))
@printf("  Iterações = %d\n", length(h_gd)-1)

println()
print_hist("1.1(b) NEWTON'S METHOD — Backtracking (t₀=1, α=0.4, β=0.5)", h_nt; show_all=true)
@printf("  x*        = [%.6f, %.6f, %.6f]\n", x_nt...)
@printf("  f(x*)     = %.6f\n", f(x_nt))
@printf("  ||∇f(x*)|| = %.2e\n", norm(grad_f(x_nt)))
@printf("  Iterações = %d\n", length(h_nt)-1)

println("\n"*"="^65)
println("  COMPARAÇÃO: Gradient Descent vs Newton")
println("="^65)
@printf("  %-16s %20s %20s\n", "", "Grad. Descent", "Newton")
println("-"^65)
@printf("  %-16s %20d %20d\n", "Iterações", length(h_gd)-1, length(h_nt)-1)
@printf("  %-16s %20.6f %20.6f\n", "f(x*)", f(x_gd), f(x_nt))
@printf("  %-16s %20.2e %20.2e\n", "||∇f(x*)||", norm(grad_f(x_gd)), norm(grad_f(x_nt)))

  1.1(a) GRADIENT DESCENT — Backtracking (t₀=1, α=0.4, β=0.5)
  Ponto inicial : [1.0000, 1.0000, 2.0000]
-----------------------------------------------------------------
   Iter            f(x)       ||∇f(x)||
-----------------------------------------------------------------
      0        4.035016        7.04e-01
      1        3.909290        5.46e-02
      2        3.908265        2.01e-02
      3        3.908130        6.70e-03
      4        3.908115        2.15e-03
      5        3.908114        6.80e-04
      6        3.908114        2.14e-04
      7        3.908114        6.74e-05
      8        3.908114        2.12e-05
      9        3.908114        1.39e-05
     10        3.908114        9.16e-06
-----------------------------------------------------------------
  x*        = [0.926210, 0.926210, 1.653421]
  f(x*)     = 3.908114
  ||∇f(x*)|| = 9.16e-06
  Iterações = 10

  1.1(b) NEWTON'S METHOD — Backtracking (t₀=1, α=0.4, β=0.5)
  Ponto inicial : [1.0000, 1.0000, 2.0000]
---

---
## 1.2 — Método BFGS (Quasi-Newton)

Mesma função $f$. $H_0 = I$. Se $y_k^T s_k < \varepsilon = 10^{-5}$, fixa-se $\rho_k = 1/\varepsilon$.

In [5]:
using LinearAlgebra, Printf

# ---------- Função e utilitários (redefinidos para célula autocontida) ----------
f(x) = x[3]*log(exp(x[1]/x[3]) + exp(x[2]/x[3])) + (x[3]-2)^2 + exp(1/(x[1]+x[2]))
in_dom(x) = (x[1]+x[2]) > 1e-10 && x[3] > 1e-10

function grad_f(x; h=1e-5)
    fx = f(x); g = zeros(3)
    for i in 1:3; xp = copy(x); xp[i] += h; g[i] = (f(xp)-fx)/h; end
    return g
end

function backtrack(x, d, g; t0=1.0, α=0.4, β=0.5)
    t, fx, gd = t0, f(x), dot(g,d)
    for _ in 1:100
        xn = x + t*d
        in_dom(xn) && f(xn) <= fx + α*t*gd && return t
        t *= β
    end
    return t
end

# ---------- BFGS ----------
function bfgs_method(x0; tol=1e-5, maxiter=10000, ε_ys=1e-5)
    n = length(x0); x = copy(x0)
    Hk = Matrix{Float64}(I, n, n)
    g = grad_f(x)
    hist = [(copy(x), f(x), norm(g))]

    for k in 1:maxiter
        norm(g) < tol && break
        d = -Hk * g
        if dot(g,d) >= 0; d = -g; Hk = Matrix{Float64}(I,n,n); end
        t = backtrack(x, d, g)
        x_new = x + t*d; g_new = grad_f(x_new)
        sk = x_new - x; yk = g_new - g; ys = dot(yk, sk)
        ρ = ys > ε_ys ? 1.0/ys : 1.0/ε_ys
        V = I - ρ*sk*yk'
        Hk = V * Hk * V' + ρ*sk*sk'
        x, g = x_new, g_new
        push!(hist, (copy(x), f(x), norm(g)))
    end
    return x, hist
end

# ---------- Execução ----------
x0 = [1.0, 1.0, 2.0]
x_bf, h_bf = bfgs_method(x0)

println("="^65)
println("  1.2 MÉTODO BFGS (Quasi-Newton) — H₀ = I")
println("="^65)
@printf("  Ponto inicial : [%.4f, %.4f, %.4f]\n", x0...)
@printf("  Tolerância    : 1e-5 (norma do gradiente)\n")
@printf("  ε (threshold yᵀs) : 1e-5\n")
println("-"^65)
@printf("  %5s  %14s  %14s\n", "Iter", "f(x)", "||∇f(x)||")
println("-"^65)
if length(h_bf) <= 12
    for (i,h) in enumerate(h_bf)
        @printf("  %5d  %14.6f  %14.2e\n", i-1, h[2], h[3])
    end
else
    for i in 1:5
        @printf("  %5d  %14.6f  %14.2e\n", i-1, h_bf[i][2], h_bf[i][3])
    end
    println("     ...")
    for i in max(6,length(h_bf)-2):length(h_bf)
        @printf("  %5d  %14.6f  %14.2e\n", i-1, h_bf[i][2], h_bf[i][3])
    end
end
println("-"^65)
@printf("  x*        = [%.6f, %.6f, %.6f]\n", x_bf...)
@printf("  f(x*)     = %.6f\n", f(x_bf))
@printf("  ||∇f(x*)|| = %.2e\n", norm(grad_f(x_bf)))
@printf("  Iterações = %d\n", length(h_bf)-1)

  1.2 MÉTODO BFGS (Quasi-Newton) — H₀ = I
  Ponto inicial : [1.0000, 1.0000, 2.0000]
  Tolerância    : 1e-5 (norma do gradiente)
  ε (threshold yᵀs) : 1e-5
-----------------------------------------------------------------
   Iter            f(x)       ||∇f(x)||
-----------------------------------------------------------------
      0        4.035016        7.04e-01
      1        3.909290        5.46e-02
      2        3.908259        1.97e-02
      3        3.908114        1.23e-03
      4        3.908114        2.58e-05
      5        3.908114        1.40e-05
      6        3.908114        1.08e-05
      7        3.908114        1.02e-05
      8        3.908114        9.89e-06
-----------------------------------------------------------------
  x*        = [0.926211, 0.926211, 1.653421]
  f(x*)     = 3.908114
  ||∇f(x*)|| = 9.89e-06
  Iterações = 8


---
## 1.3 — Reator CSTR: Formulação e Solução LP

**Objetivo:** $\min \sum_{k=0}^{3}(|y_1(k)-c_{sp}|+|y_3(k)-h_{sp}|)$

**Linearização:** $|z| \le t \Leftrightarrow z \le t,\; -z \le t,\; t \ge 0$

**Dinâmica:** $x(k+1) = Ax(k) + Bu(k)$, $y(k) = x(k)$

In [6]:
using JuMP, HiGHS, LinearAlgebra, Printf

# ---------- Dados do problema ----------
A = [ 0.2681  -0.00338  -0.00728;
      9.703    0.3279   -25.44;
      0.0      0.0       1.0]

B = [-0.00537   0.1655;
      1.297    97.91;
      0.0      -6.637]

x0  = [-0.03, 0.0, 0.3]   # estado inicial
csp = 0.0;  hsp = 0.0      # setpoints
N   = 3                     # horizonte (k = 0,1,2,3)

xlb = [-0.05, -5.0, -0.5];  xub = [ 0.05,  5.0,  0.5]
ulb = [-10.0, -0.05];       uub = [ 10.0,  0.05]

# ---------- Formulação LP ----------
model = Model(HiGHS.Optimizer); set_silent(model)

@variable(model, xlb[i] <= x[i=1:3, k=0:N] <= xub[i])
@variable(model, ulb[j] <= u[j=1:2, k=0:N-1] <= uub[j])
@variable(model, t1[k=0:N] >= 0)  # auxiliar para |x1 - csp|
@variable(model, t3[k=0:N] >= 0)  # auxiliar para |x3 - hsp|

@constraint(model, [i=1:3], x[i,0] == x0[i])                              # condição inicial
for k in 0:N-1, i in 1:3                                                   # dinâmica
    @constraint(model, x[i,k+1] == sum(A[i,j]*x[j,k] for j in 1:3) + sum(B[i,j]*u[j,k] for j in 1:2))
end
for k in 0:N                                                                # valor absoluto
    @constraint(model,  (x[1,k] - csp) <= t1[k])
    @constraint(model, -(x[1,k] - csp) <= t1[k])
    @constraint(model,  (x[3,k] - hsp) <= t3[k])
    @constraint(model, -(x[3,k] - hsp) <= t3[k])
end
@objective(model, Min, sum(t1[k] + t3[k] for k in 0:N))

# ---------- Resolução ----------
optimize!(model)

# ---------- Resultados ----------
println("="^70)
println("  1.3 REATOR CSTR — PROBLEMA LP (JuMP + HiGHS)")
println("="^70)
@printf("  Status         : %s\n", termination_status(model))
@printf("  Valor ótimo J* : %.6f\n", objective_value(model))

println("\n  Trajetória dos estados x(k):")
println("  "*"-"^58)
@printf("  %3s  %14s  %14s  %14s\n", "k", "x₁ (c-cˢ)", "x₂ (T-Tˢ)", "x₃ (h-hˢ)")
println("  "*"-"^58)
for k in 0:N
    @printf("  %3d  %14.6f  %14.6f  %14.6f\n", k, value(x[1,k]), value(x[2,k]), value(x[3,k]))
end

println("\n  Entradas de controle u(k):")
println("  "*"-"^40)
@printf("  %3s  %14s  %14s\n", "k", "u₁ (Tc-Tcˢ)", "u₂ (F-Fˢ)")
println("  "*"-"^40)
for k in 0:N-1
    @printf("  %3d  %14.6f  %14.6f\n", k, value(u[1,k]), value(u[2,k]))
end

println("\n  Desvios dos setpoints em cada instante:")
println("  "*"-"^50)
@printf("  %3s  %14s  %14s  %14s\n", "k", "|c - csp|", "|h - hsp|", "Soma")
println("  "*"-"^50)
for k in 0:N
    dc = abs(value(x[1,k]) - csp); dh = abs(value(x[3,k]) - hsp)
    @printf("  %3d  %14.6f  %14.6f  %14.6f\n", k, dc, dh, dc+dh)
end

println("\n  Verificação da dinâmica (erro = ||Ax(k)+Bu(k) - x(k+1)||):")
xv = hcat([value.(x[:,k]) for k in 0:N]...)
uv = hcat([value.(u[:,k]) for k in 0:N-1]...)
for k in 0:N-1
    err = norm(A*xv[:,k+1] + B*uv[:,k+1] - xv[:,k+2])
    @printf("    k=%d → k=%d : %.2e\n", k, k+1, err)
end

  1.3 REATOR CSTR — PROBLEMA LP (JuMP + HiGHS)
  Status         : OPTIMAL
  Valor ótimo J* : 0.330000

  Trajetória dos estados x(k):
  ----------------------------------------------------------
    k       x₁ (c-cˢ)       x₂ (T-Tˢ)       x₃ (h-hˢ)
  ----------------------------------------------------------
    0       -0.030000       -0.000000        0.300000
    1       -0.000000       -4.160730       -0.000000
    2       -0.000000        2.032355       -0.000000
    3       -0.000000       -0.992727       -0.000000

  Entradas de controle u(k):
  ----------------------------------------
    k     u₁ (Tc-Tcˢ)       u₂ (F-Fˢ)
  ----------------------------------------
    0       -0.511399        0.045201
    1        2.618858       -0.000000
    2       -1.279211       -0.000000

  Desvios dos setpoints em cada instante:
  --------------------------------------------------
    k       |c - csp|       |h - hsp|            Soma
  --------------------------------------------------
   